In [1]:
"""
Step 7: two honesty checks on whether the dual-filter front end beats plain boxcar.

CHECK A (5-qubit synthetic bus): retune the boxcar window on the bus for each crosstalk strength, then redo the
        boxcar-vs-dual-filter comparison. Previously the boxcar used a fixed window [50,500) tuned on clean single-qubit
        data, which may have handicapped it.
CHECK B (single-qubit, real data + added noise): add white Gaussian noise to the QICK traces to lower the SNR, and at
        each noise level re-tune boxcar (window) and re-calibrate the filters (WMF/RMF, best T). Shows whether the
        dual filter's advantage grows as SNR falls.

Both checks use paired comparisons with standard errors. Check A is SYNTHETIC; check B starts from real data but the
added noise is white and artificial.
"""
import os
import numpy as np

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
CHECKS = ("A", "B")
SEED = 42
POOL_TRAIN_EACH = 100_000
N_SAMPLES = 770

# check A (bus)
N_QUBITS = 5
FS = 500.0e6
IF_FREQS = np.array([64.729e6, 25.366e6, -24.790e6, -70.269e6, -127.282e6])
T_BINS = 500
EMA_ALPHA = 0.05
RIDGE_LAMBDA_REL = 1e-4
CLIP = 99.99
N_CAL, N_CLF, N_TEST = 30_000, 50_000, 100_000
STRENGTHS = [0.0, 1.0, 2.0]
PREV_GAP = {0.0: 0.54, 1.0: 1.79, 2.0: 5.51}       # df_cross - bx_cross with the fixed [50,500) window (step 5c)

# check B (single qubit + added noise)
NOISE_K = [0, 0.5, 1, 2, 4]                         # added noise std = k * measured original noise std
B_N_CAL, B_N_CLF = 50_000, 100_000
B_TS = [250, 385, 500, 650, 770]
RIDGE_LAMBDA = 1e-3


def find_file(fname):
    for d in [DATA_DIR, os.path.join(DATA_DIR, "python"), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(f"Cannot find {fname} in {DATA_DIR} or current folder.")


# ------------------------------------------------------------------ source pools
Xtr = np.load(find_file("0528_X_train_0_770.npy"), mmap_mode="r")
Xte_m = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(find_file("0528_y_train_0_770.npy")).astype(int)
yte = np.load(find_file("0528_y_test_0_770.npy")).astype(int)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ytr))
cal_pool = np.sort(perm[:POOL_TRAIN_EACH])
clf_pool = np.sort(perm[POOL_TRAIN_EACH:2 * POOL_TRAIN_EACH])


def load_pool(X, idx):
    out = np.empty((len(idx), 2 * N_SAMPLES), np.int16)
    for i in range(0, len(idx), 20_000):
        out[i:i + 20_000] = np.rint(X[idx[i:i + 20_000]]).astype(np.int16)
    return out


print("loading source-shot pools ...")
S_cal, y_cal_pool = load_pool(Xtr, cal_pool), ytr[cal_pool]
S_clf, y_clf_pool = load_pool(Xtr, clf_pool), ytr[clf_pool]
S_te, y_te_pool = load_pool(Xte_m, np.arange(len(yte))), yte


# ------------------------------------------------------------------ shared helpers
def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def fid_pred(pred, y):
    return 0.5 * (np.mean(~pred[y == 0]) + np.mean(pred[y == 1]))


def lda_fit(F, y):
    m0, m1 = F[y == 0].mean(0), F[y == 1].mean(0)
    Sw = 0.5 * (np.cov(F[y == 0].T) + np.cov(F[y == 1].T))
    return np.linalg.solve(Sw, m1 - m0)


def ema_1d(x, alpha):
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = alpha * x[t] + (1.0 - alpha) * out[t - 1]
    return out


# ================================================================== CHECK A
if "A" in CHECKS:
    print("\n################ CHECK A: retuned boxcar window on the 5-qubit synthetic bus ################")
    idx_cal = rng.integers(0, len(y_cal_pool), size=(N_CAL, N_QUBITS))
    idx_clf = rng.integers(0, len(y_clf_pool), size=(N_CLF, N_QUBITS))
    idx_te = rng.integers(0, len(y_te_pool), size=(N_TEST, N_QUBITS))
    Y_cal, Y_clf, Y_te = y_cal_pool[idx_cal], y_clf_pool[idx_clf], y_te_pool[idx_te]

    t_sec = np.arange(T_BINS, dtype=np.float64) / FS
    carriers = np.exp(1j * 2.0 * np.pi * IF_FREQS[:, None] * t_sec[None, :])
    LIN = np.array([[1.00, 0.15, 0.04, 0.01, 0.00], [0.15, 1.00, 0.16, 0.04, 0.01], [0.04, 0.16, 1.00, 0.14, 0.04],
                    [0.01, 0.04, 0.14, 1.00, 0.15], [0.00, 0.01, 0.04, 0.15, 1.00]])
    PH = np.array([[0.00, 0.28, 0.08, 0.00, 0.00], [-0.26, 0.00, 0.30, 0.08, 0.00], [0.07, -0.28, 0.00, 0.26, 0.07],
                   [0.00, 0.07, -0.25, 0.00, 0.28], [0.00, 0.00, 0.07, -0.27, 0.00]])

    def synth(S, idx, Y, s, chunk=5000):
        n = len(idx)
        L = np.eye(N_QUBITS) + s * (LIN - np.eye(N_QUBITS))
        P = s * PH
        I_out = np.empty((n, T_BINS), np.int16)
        Q_out = np.empty((n, T_BINS), np.int16)
        for a in range(0, n, chunk):
            b = min(a + chunk, n)
            bb = np.empty((b - a, N_QUBITS, T_BINS), np.complex128)
            for q in range(N_QUBITS):
                rows = S[idx[a:b, q], :2 * T_BINS].astype(np.float64)
                bb[:, q, :] = rows[:, 0::2] + 1j * rows[:, 1::2]
            sgn = 2.0 * Y[a:b] - 1.0
            theta = sgn @ P.T
            prev = np.zeros_like(sgn)
            prev[:, 1:] = sgn[:, :-1]
            bb = bb * ((1.0 + 0.12 * s * prev)[:, :, None] * np.exp(1j * theta[:, :, None]))
            bb = np.einsum("pq,bqt->bpt", L, bb)
            bus = np.einsum("bkt,kt->bt", bb, carriers) / np.sqrt(N_QUBITS)
            I_out[a:b] = np.clip(np.rint(bus.real), -32768, 32767).astype(np.int16)
            Q_out[a:b] = np.clip(np.rint(bus.imag), -32768, 32767).astype(np.int16)
        return I_out, Q_out

    def premix(q, wmI, wmQ, wrI, wrQ, W):
        c, sn = np.cos(2.0 * np.pi * IF_FREQS[q] * t_sec), np.sin(2.0 * np.pi * IF_FREQS[q] * t_sec)
        W[q, 0, 0], W[q, 0, 1] = wmI * c, wmI * sn
        W[q, 1, 0], W[q, 1, 1] = -wmQ * sn, wmQ * c
        W[q, 2, 0], W[q, 2, 1] = wrI * c, wrI * sn
        W[q, 3, 0], W[q, 3, 1] = -wrQ * sn, wrQ * c

    def box_tables(a, b):
        rect = np.zeros(T_BINS)
        rect[a:b] = 1.0
        W = np.zeros((N_QUBITS, 4, 2, T_BINS))
        z = np.zeros(T_BINS)
        for q in range(N_QUBITS):
            premix(q, rect, rect, z, z, W)
        return W

    def df_tables(I_raw, Q_raw, Y):
        S_raw = I_raw.astype(np.float64) + 1j * Q_raw.astype(np.float64)
        W = np.zeros((N_QUBITS, 4, 2, T_BINS))
        for q in range(N_QUBITS):
            Zq = S_raw * np.exp(-1j * 2.0 * np.pi * IF_FREQS[q] * t_sec)[None, :]
            Iq, Qq, yq = Zq.real, Zq.imag, Y[:, q]
            m0, m1 = yq == 0, yq == 1
            mI0, mQ0, vI0, vQ0 = Iq[m0].mean(0), Qq[m0].mean(0), Iq[m0].var(0), Qq[m0].var(0)
            mI1, mQ1, vI1, vQ1 = Iq[m1].mean(0), Qq[m1].mean(0), Iq[m1].var(0), Qq[m1].var(0)
            mtv = np.c_[Iq.mean(1), Qq.mean(1)]
            C0, C1 = mtv[m0].mean(0), mtv[m1].mean(0)
            relax = m1 & (np.linalg.norm(mtv - C0, axis=1) < 0.5 * np.linalg.norm(C0 - C1))
            wmI = ema_1d(mI1 - mI0, EMA_ALPHA) / (0.5 * (vI0 + vI1))
            wmQ = ema_1d(mQ1 - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQ1))
            if relax.sum() >= 20:
                mIR, mQR, vIR, vQR = Iq[relax].mean(0), Qq[relax].mean(0), Iq[relax].var(0), Qq[relax].var(0)
                wrI = ema_1d(mIR - mI0, EMA_ALPHA) / (0.5 * (vI0 + vIR))
                wrQ = ema_1d(mQR - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQR))
            else:
                wrI = wrQ = np.zeros(T_BINS)
            premix(q, wmI, wmQ, wrI, wrQ, W)
        return W

    def accumulate(I_raw, Q_raw, W, chunk=5000):
        N = len(I_raw)
        out = np.empty((N, N_QUBITS, 4))
        WI = W[:, :, 0, :].reshape(N_QUBITS * 4, T_BINS).T
        WQ = W[:, :, 1, :].reshape(N_QUBITS * 4, T_BINS).T
        for s_ in range(0, N, chunk):
            e = min(s_ + chunk, N)
            out[s_:e] = (I_raw[s_:e].astype(np.float64) @ WI + Q_raw[s_:e].astype(np.float64) @ WQ).reshape(e - s_, N_QUBITS, 4)
        return out

    def normalize(Ac, At):
        mu = Ac.mean(0)
        A = np.percentile(np.abs(Ac - mu), CLIP, axis=0)
        n = np.ceil(np.log2(np.maximum(A, 1e-9)))
        f = lambda a: np.clip((a + 2.0 ** n - mu) / 2.0 ** (n + 1), 0, 1) - 0.5
        return f(Ac), f(At)

    def quad_cross(V, q, nf):
        vk = V[:, q, :nf]
        self_q = [vk[:, i] * vk[:, j] for i in range(nf) for j in range(i, nf)]
        cross = []
        for j in range(N_QUBITS):
            if j != q:
                vj = V[:, j, :]
                cross += [vk[:, 0] * vj[:, 0], vk[:, 0] * vj[:, 1], vk[:, 1] * vj[:, 0], vk[:, 1] * vj[:, 1]]
        return np.stack(self_q, 1), np.stack(cross, 1)

    def features(arch, Vbx, Vdf, q):
        if arch == "bx_single":
            return Vbx[:, q, :2]
        if arch == "bx_lin10":
            return Vbx[:, :, :2].reshape(len(Vbx), -1)
        if arch == "bx_cross":
            sq, cr = quad_cross(Vbx, q, 2)
            return np.c_[Vbx[:, :, :2].reshape(len(Vbx), -1), sq, cr]
        if arch == "df_single":
            return Vdf[:, q, :]
        if arch == "df_lin20":
            return Vdf.reshape(len(Vdf), -1)
        if arch == "df_cross":
            sq, cr = quad_cross(Vdf, q, 4)
            return np.c_[Vdf.reshape(len(Vdf), -1), sq, cr]
        raise ValueError(arch)

    def demod_cumsums(I_raw, Q_raw, chunk=2000):
        N = len(I_raw)
        cI = np.zeros((N, N_QUBITS, T_BINS + 1), np.float32)
        cQ = np.zeros((N, N_QUBITS, T_BINS + 1), np.float32)
        dem = np.exp(-1j * 2.0 * np.pi * IF_FREQS[:, None] * t_sec[None, :])
        for s_ in range(0, N, chunk):
            e = min(s_ + chunk, N)
            z = (I_raw[s_:e].astype(np.float64) + 1j * Q_raw[s_:e].astype(np.float64))[:, None, :] * dem[None]
            cI[s_:e, :, 1:] = np.cumsum(z.real, axis=2)
            cQ[s_:e, :, 1:] = np.cumsum(z.imag, axis=2)
        return cI, cQ

    def tune_window(cI, cQ, Y):
        best = None
        for a in range(0, 151, 25):
            for b in range(250, T_BINS + 1, 25):
                fs = []
                for q in range(N_QUBITS):
                    F = np.c_[cI[:, q, b] - cI[:, q, a], cQ[:, q, b] - cQ[:, q, a]].astype(np.float64)
                    s_ = F @ lda_fit(F, Y[:, q])
                    fs.append(fid_pred(s_ > best_threshold(s_, Y[:, q]), Y[:, q]))
                if best is None or np.mean(fs) > best[0]:
                    best = (np.mean(fs), a, b)
        return best

    ARCHS = ["bx_single", "bx_lin10", "bx_cross", "df_single", "df_lin20", "df_cross"]
    summary_A = []
    for s in STRENGTHS:
        print(f"\n=== crosstalk strength s = {s} ({'CONTROL: no crosstalk' if s == 0 else 'x original model'}) ===")
        I_cal, Q_cal = synth(S_cal, idx_cal, Y_cal, s)
        I_clf, Q_clf = synth(S_clf, idx_clf, Y_clf, s)
        I_te, Q_te = synth(S_te, idx_te, Y_te, s)

        cI, cQ = demod_cumsums(I_clf, Q_clf)
        _, a_best, b_best = tune_window(cI, cQ, Y_clf)
        del cI, cQ
        print(f"tuned boxcar window on the bus: [{a_best}, {b_best})   (previous fixed window: [50, 500))")

        W_box, W_df = box_tables(a_best, b_best), df_tables(I_cal, Q_cal, Y_cal)
        Vdf_c, Vdf_t = normalize(accumulate(I_clf, Q_clf, W_df), accumulate(I_te, Q_te, W_df))
        Vbx_c, Vbx_t = normalize(accumulate(I_clf, Q_clf, W_box), accumulate(I_te, Q_te, W_box))

        corr, fids = {}, {}
        for arch in ARCHS:
            corr[arch] = np.zeros(Y_te.shape, bool)
            fids[arch] = []
            for q in range(N_QUBITS):
                Fc = np.c_[np.ones(len(Vdf_c)), features(arch, Vbx_c, Vdf_c, q)]
                Ft = np.c_[np.ones(len(Vdf_t)), features(arch, Vbx_t, Vdf_t, q)]
                w = np.linalg.solve(Fc.T @ Fc + RIDGE_LAMBDA_REL * len(Fc) * np.eye(Fc.shape[1]),
                                    Fc.T @ (2.0 * Y_clf[:, q] - 1.0))
                pq = (Ft @ w) > best_threshold(Fc @ w, Y_clf[:, q])
                corr[arch][:, q] = pq == (Y_te[:, q] == 1)
                fids[arch].append(fid_pred(pq, Y_te[:, q]))
        base = corr["bx_single"].mean(1)
        print(f"{'architecture':<12s} {'mean F':>7s} {'gain vs bx_single (pts)':>25s}")
        for arch in ARCHS:
            d = corr[arch].mean(1) - base
            print(f"{arch:<12s} {np.mean(fids[arch]):7.4f} {100 * d.mean():+10.2f} +/- {100 * d.std() / np.sqrt(len(d)):.2f}")
        d = corr["df_cross"].mean(1) - corr["bx_cross"].mean(1)
        gap, se = 100 * d.mean(), 100 * d.std() / np.sqrt(len(d))
        print(f"df_cross minus bx_cross (retuned window): {gap:+.2f} +/- {se:.2f} points   "
              f"[previous fixed window: {PREV_GAP.get(s, float('nan')):+.2f}]")
        summary_A.append((s, a_best, b_best, gap, se))

    print("\n----- CHECK A summary: dual-filter advantage over boxcar (with cross-terms) -----")
    print(f"{'strength':>8s} {'window':>10s} {'gap (pts)':>16s} {'previous':>9s}")
    for s, a, b, gap, se in summary_A:
        print(f"{s:8.1f} {f'[{a},{b})':>10s} {gap:+8.2f} +/- {se:.2f} {PREV_GAP.get(s, float('nan')):+9.2f}")
    print("If the retuned gap is much smaller than 'previous', the old comparison handicapped boxcar.")


# ================================================================== CHECK B
if "B" in CHECKS:
    print("\n################ CHECK B: SNR sweep on single-qubit data (added white noise) ################")
    CH = 20_000
    Xcal0 = S_cal[:B_N_CAL]
    ycal = y_cal_pool[:B_N_CAL]
    Xclf0 = S_clf[:B_N_CLF]
    yclf = y_clf_pool[:B_N_CLF]
    Xte0 = S_te
    ytest = y_te_pool

    # measured original noise std (within-class spread per bin, late bins where the signal has settled)
    cls0 = Xclf0[yclf == 0][:20_000].astype(np.float64)
    var_I = cls0[:, 2 * 300:2 * 700:2].var(0).mean()
    var_Q = cls0[:, 2 * 300 + 1:2 * 700:2].var(0).mean()
    sigma0 = float(np.sqrt(0.5 * (var_I + var_Q)))
    print(f"measured original per-sample noise std ~ {sigma0:.1f} ADC counts (per quadrature)")

    def noisy(X, sd):
        out = X.astype(np.float32)
        if sd > 0:
            for i in range(0, len(out), CH):
                out[i:i + CH] += sd * rng.standard_normal(out[i:i + CH].shape, dtype=np.float32)
        return out

    def calibrate_single(Xc, yc):
        I, Q = Xc[:, 0::2].astype(np.float64), Xc[:, 1::2].astype(np.float64)
        m0, m1 = yc == 0, yc == 1
        S0 = (I[m0].mean(0), Q[m0].mean(0), I[m0].var(0), Q[m0].var(0))
        S1 = (I[m1].mean(0), Q[m1].mean(0), I[m1].var(0), Q[m1].var(0))
        mtv = np.c_[I.mean(1), Q.mean(1)]
        C0, C1 = mtv[m0].mean(0), mtv[m1].mean(0)
        relax = m1 & (np.linalg.norm(mtv - C0, axis=1) < 0.5 * np.linalg.norm(C0 - C1))
        mk = lambda A, B: [ema_1d(B[k] - A[k], EMA_ALPHA) / (0.5 * (A[2 + k] + B[2 + k])) for k in (0, 1)]
        wmf = mk(S0, S1)
        if relax.sum() >= 50:
            SR = (I[relax].mean(0), Q[relax].mean(0), I[relax].var(0), Q[relax].var(0))
            wrmf = mk(S0, SR)
        else:
            wrmf = [np.zeros(N_SAMPLES), np.zeros(N_SAMPLES)]
        return [wmf[0], wmf[1], wrmf[0], wrmf[1]], int(relax.sum())

    def cum4(X, W):
        out = np.empty((len(X), len(B_TS), 4))
        cols = np.array(B_TS) - 1
        for i in range(0, len(X), CH):
            x = X[i:i + CH]
            I, Q = x[:, 0::2], x[:, 1::2]
            parts = [I * W[0].astype(np.float32), Q * W[1].astype(np.float32),
                     I * W[2].astype(np.float32), Q * W[3].astype(np.float32)]
            for k, pk in enumerate(parts):
                out[i:i + CH, :, k] = np.cumsum(pk, axis=1, dtype=np.float64)[:, cols]
        return out

    def cumsums(X):
        cI = np.zeros((len(X), N_SAMPLES + 1))
        cQ = np.zeros((len(X), N_SAMPLES + 1))
        for i in range(0, len(X), CH):
            x = X[i:i + CH]
            cI[i:i + CH, 1:] = np.cumsum(x[:, 0::2], axis=1, dtype=np.float64)
            cQ[i:i + CH, 1:] = np.cumsum(x[:, 1::2], axis=1, dtype=np.float64)
        return cI, cQ

    def ridge_fit(F, y, lam=RIDGE_LAMBDA):
        mu, sd = F.mean(0), F.std(0) + 1e-12
        Z = np.c_[np.ones(len(F)), (F - mu) / sd]
        return mu, sd, np.linalg.solve(Z.T @ Z + lam * np.eye(Z.shape[1]), Z.T @ (2.0 * y - 1.0))

    def ridge_score(F, m):
        mu, sd, w = m
        return np.c_[np.ones(len(F)), (F - mu) / sd] @ w

    def full_quad(Z):
        return np.c_[Z, np.stack([Z[:, i] * Z[:, j] for i in range(4) for j in range(i, 4)], 1)]

    rows = []
    for k in NOISE_K:
        sd = k * sigma0
        print(f"\n--- added noise std = {k} x original ({sd:.0f} counts); noise power x{1 + k * k:.2f} ---")
        Xc, Xl, Xt = noisy(Xcal0, sd), noisy(Xclf0, sd), noisy(Xte0, sd)

        # tuned boxcar
        cIl, cQl = cumsums(Xl)
        cIt, cQt = cumsums(Xt)
        best = None
        for a in range(0, 301, 25):
            for b in range(a + 50, N_SAMPLES + 1, 20):
                F = np.c_[cIl[:, b] - cIl[:, a], cQl[:, b] - cQl[:, a]]
                s_ = F @ lda_fit(F, yclf)
                f = fid_pred(s_ > best_threshold(s_, yclf), yclf)
                if best is None or f > best[0]:
                    best = (f, a, b)
        _, a_b, b_b = best
        Fl = np.c_[cIl[:, b_b] - cIl[:, a_b], cQl[:, b_b] - cQl[:, a_b]]
        Ft = np.c_[cIt[:, b_b] - cIt[:, a_b], cQt[:, b_b] - cQt[:, a_b]]
        w = lda_fit(Fl, yclf)
        pred_bx = (Ft @ w) > best_threshold(Fl @ w, yclf)
        corr_bx = pred_bx == (ytest == 1)
        F_bx = fid_pred(pred_bx, ytest)
        del cIl, cQl, cIt, cQt

        # dual filter (recalibrated on the noisy calibration set), best T chosen on the CLF set
        W, n_relax = calibrate_single(Xc, ycal)
        Fc_all, Ft_all = cum4(Xl, W), cum4(Xt, W)
        res = {}
        for kind in ("lin4", "full quad"):
            best_k = None
            for j, T in enumerate(B_TS):
                mu_b, sd_b = Fc_all[:, j].mean(0), Fc_all[:, j].std(0) + 1e-12
                Zc, Zt = (Fc_all[:, j] - mu_b) / sd_b, (Ft_all[:, j] - mu_b) / sd_b
                if kind == "full quad":
                    Zc, Zt = full_quad(Zc), full_quad(Zt)
                m = ridge_fit(Zc, yclf)
                thr = best_threshold(ridge_score(Zc, m), yclf)
                f_clf = fid_pred(ridge_score(Zc, m) > thr, yclf)
                if best_k is None or f_clf > best_k[0]:
                    pred = ridge_score(Zt, m) > thr
                    best_k = (f_clf, T, pred)
            pred = best_k[2]
            d = (pred == (ytest == 1)).astype(float) - corr_bx.astype(float)
            res[kind] = (fid_pred(pred, ytest), best_k[1], 100 * d.mean(), 100 * d.std() / np.sqrt(len(d)))
        rows.append((k, 1 + k * k, F_bx, a_b, b_b, res, n_relax))
        print(f"  boxcar tuned [{a_b},{b_b}): F={F_bx:.4f}   RMF relaxation traces found: {n_relax}")
        for kind, (F, T, g, se) in res.items():
            print(f"  {kind:<10s} (best T={T}): F={F:.4f}   gain vs boxcar {g:+.2f} +/- {se:.2f} points")

    print("\n----- CHECK B summary: gain of the dual-filter designs over tuned boxcar (points) -----")
    print(f"{'noise k':>7s} {'power x':>8s} {'boxcar F':>9s} {'window':>10s} {'lin4 gain':>16s} {'full quad gain':>18s}")
    for k, pw, F_bx, a_b, b_b, res, _ in rows:
        l, q = res["lin4"], res["full quad"]
        print(f"{k:7.1f} {pw:8.2f} {F_bx:9.4f} {f'[{a_b},{b_b})':>10s} {l[2]:+8.2f}+/-{l[3]:.2f} {q[2]:+10.2f}+/-{q[3]:.2f}")
    print("If the gains grow as noise grows, your design shines at LOW SNR. If they stay near 0, tuned boxcar is enough.")
    print("Note: added noise is white and artificial; real low-SNR data may behave differently.")

loading source-shot pools ...

################ CHECK A: retuned boxcar window on the 5-qubit synthetic bus ################

=== crosstalk strength s = 0.0 (CONTROL: no crosstalk) ===
tuned boxcar window on the bus: [75, 500)   (previous fixed window: [50, 500))
architecture  mean F   gain vs bx_single (pts)
bx_single     0.8784      +0.00 +/- 0.00
bx_lin10      0.8784      -0.01 +/- 0.01
bx_cross      0.8784      -0.00 +/- 0.01
df_single     0.8819      +0.34 +/- 0.02
df_lin20      0.8819      +0.34 +/- 0.02
df_cross      0.8817      +0.33 +/- 0.02
df_cross minus bx_cross (retuned window): +0.33 +/- 0.03 points   [previous fixed window: +0.54]

=== crosstalk strength s = 1.0 (x original model) ===
tuned boxcar window on the bus: [100, 500)   (previous fixed window: [50, 500))
architecture  mean F   gain vs bx_single (pts)
bx_single     0.8495      +0.00 +/- 0.00
bx_lin10      0.8579      +0.84 +/- 0.03
bx_cross      0.8616      +1.22 +/- 0.03
df_single     0.8574      +0.80 +/- 0.03
